# Flujo de Ejecucion de un Agente (Emulacion de smolagents)

Este notebook emula manualmente el ciclo interno que sigue un agente como los de la libreria `smolagents`.
El objetivo es entender **que ocurre por dentro** cuando un agente decide usar una herramienta.

## El ciclo de un agente en tres pasos

```
  LLM recibe el system prompt
         |
         v
  LLM genera una accion (Thought + Action)
         |
         v
  El motor del agente ejecuta la herramienta
         |
         v
  El resultado (Observation) se devuelve al LLM
         |
         v
  Se repite hasta que el LLM emite una respuesta final
```

Cada seccion de este notebook corresponde a una pieza del ciclo.

## Paso 1 — La Clase Base `Tool` (el contrato)

En `smolagents`, toda herramienta hereda de una clase base que impone una interfaz comun.
Esto le permite al motor del agente tratar cualquier herramienta de forma uniforme,
sin importar lo que haga por dentro.


Los cuatro atributos obligatorios son:

| Atributo | Proposito |
|---|---|
| `name` | Identificador unico que el LLM usara para invocar la herramienta |
| `description` | Texto en lenguaje natural que el LLM lee para saber *que* hace la herramienta |
| `inputs` | Esquema de los argumentos que acepta (tipo y descripcion de cada uno) |
| `output_type` | Tipo del valor de retorno (`"string"`, `"number"`, etc.) |

El metodo `forward` es el punto de entrada: el motor del agente lo llama cuando
el LLM decide usar la herramienta.

In [6]:
from typing import Any, Dict

# Clase madre
class Tool:
    """Clase base que define el contrato minimo para cualquier herramienta del agente."""

    name: str                           # identificador que el LLM usa para invocarla
    description: str                    # explicacion en lenguaje natural para el LLM
    inputs: Dict[str, Dict[str, Any]]   # esquema de argumentos esperados
    output_type: str                    # tipo de dato que retorna

    def forward(self, *args, **kwargs):
        """Logica real de la herramienta. Las subclases deben sobreescribir este metodo."""
        
        pass

## Paso 2 — Una Herramienta Concreta: `StockPriceTool`

Aqui creamos una herramienta real que hereda de `Tool`.
Para no depender de una API externa, usamos un diccionario `mock_data` con precios fijos.

Puntos clave a observar:

- Los **atributos de clase** (`name`, `description`, `inputs`, `output_type`) son
  los metadatos que el motor del agente leeara para construir el system prompt.
- `_log_activity` es un metodo de soporte interno (no lo ve el LLM).
- `forward` contiene la logica de negocio: buscar el simbolo, retornar el precio
  o sugerir alternativas si no existe.

In [11]:
import datetime


class StockPriceTool(Tool):
    # --- Metadatos: el LLM los lee para saber como usar esta herramienta ---
    name = "get_stock_price"
    description = "Obtiene el precio actual de una accion. Si el simbolo no existe, sugiere opciones validas."
    inputs = {
        "symbol": {
            "type": "string",
            "description": "Simbolo bursatil de la accion (ej: AAPL, TSLA, GOOGL)"
        }
    }
    output_type = "string"

    # Datos simulados: en produccion esto seria una llamada a una API financiera
    _mock_data = {"AAPL": 175.5, "TSLA": 240.2, "GOOGL": 140.1}

    def _log_activity(self, message: str) -> None:
        """Escribe un registro de auditoria en agent_log.txt (no expuesto al LLM)."""
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open("agent_log.txt", "a") as f:
            f.write(f"[{timestamp}] {message}\n")

    def forward(self, symbol: str) -> str:
        """Punto de entrada que llama el motor del agente cuando el LLM elige esta herramienta."""
        s = symbol.upper()  # normalizar a mayusculas para evitar errores de capitalizado

        if s in self._mock_data:
            result = f"El precio de {s} es {self._mock_data[s]} USD."
            self._log_activity(f"SUCCESS: {s} consultado.")
            return result
        else:
            # El LLM recibira este mensaje y podra corregir su siguiente accion
            opciones = ", ".join(self._mock_data.keys())
            error_msg = f"Error: '{s}' no encontrado. Intenta con: {opciones}"
            self._log_activity(f"ERROR: Fallo al buscar {s}.")
            return error_msg


# Prueba rapida de la herramienta de forma aislada (sin agente)
tool_solo = StockPriceTool()
print("Simbolo valido:  ", tool_solo.forward("aapl"))
print("Simbolo invalido:", tool_solo.forward("aaples"))

Simbolo valido:   El precio de AAPL es 175.5 USD.
Simbolo invalido: Error: 'AAPLES' no encontrado. Intenta con: AAPL, TSLA, GOOGL


In [8]:
precios = StockPriceTool()

In [10]:
precios.forward("AAPL")

'El precio de AAPL es 175.5 USD.'

## Paso 3 — El Motor del Agente: `MockAgent`

Este es el componente central. En `smolagents` esto es el `CodeAgent` o el `ToolCallingAgent`.
Su responsabilidad es hacer de puente entre el LLM y las herramientas.

Tiene dos responsabilidades criticas:

1. **Construir el system prompt** — Recorre todas las herramientas registradas y genera
   un texto que le explica al LLM cuales herramientas tiene disponibles y como usarlas.
   Este texto se envia al LLM al inicio de cada conversacion.

2. **Despachar la accion del LLM** — Cuando el LLM genera una accion (nombre de herramienta
   + argumentos), el motor la parsea, busca la herramienta correcta, llama a su `forward`,
   y devuelve el resultado como una `Observation` al LLM.

> **Nota:** En la vida real, el parseo de argumentos (JSON → kwargs) lo hace smolagents
> automaticamente. Aqui lo simplificamos pasando un unico string para no oscurecer el flujo.

In [12]:
class MockAgent:
    def __init__(self, tools: list):
        # Indexamos las herramientas por nombre para buscarlas en O(1)
        self.tools = {tool.name: tool for tool in tools}  # desemapcaramos la lista de herramientas en un diccionario

    def generate_system_prompt(self) -> str:
        """
        Construye el system prompt que se envia al LLM antes de la primera interaccion.
        En smolagents este texto incluye tambien reglas de formato (ReAct, JSON, etc.).
        """
        prompt = "Eres un asistente con acceso a las siguientes herramientas:\n"
        for name, tool in self.tools.items():
            prompt += f"\n- {name}: {tool.description}"
            prompt += f"\n  Argumentos: {tool.inputs}"
            prompt += f"\n  Retorna: {tool.output_type}\n"
        return prompt

    def handle_llm_call(self, tool_name: str, argument: str) -> str:
        """
        Ejecuta la accion elegida por el LLM y empaqueta el resultado como Observation.

        En el ciclo ReAct el LLM emite:
            Thought: necesito el precio de Apple
            Action: get_stock_price
            Action Input: AAPL

        Este metodo recibe esos valores y retorna la Observation que el LLM leera
        en el siguiente turno para continuar razonando.
        """
        if tool_name not in self.tools:
            return f"Error: la herramienta '{tool_name}' no existe."

        tool = self.tools[tool_name]
        print(f"[Agente] Ejecutando '{tool_name}' con argumento: '{argument}'")
        result = tool.forward(argument)

        # El prefijo 'Observation:' es la convencion ReAct para que el LLM reconozca
        # el bloque como resultado de una herramienta, no como texto generado por el.
        return f"Observation: {result}"


# Registrar las herramientas disponibles al instanciar el agente
agent = MockAgent(tools=[StockPriceTool()])

## Paso 4 — Ejecucion del Flujo Completo

Ahora conectamos todas las piezas y simulamos un turno completo del ciclo agente:

```
TURNO 1
  Usuario:  "Cual es el precio de Apple?"
  LLM piensa: debo usar get_stock_price con AAPL
  Agente:   llama a StockPriceTool.forward("AAPL")
  LLM recibe: Observation: El precio de AAPL es 175.5 USD.
  LLM responde al usuario con la respuesta final
```

In [14]:
# --- PASO A: Lo que el LLM recibe al inicio (system prompt) ---
print("=" * 55)
print("PASO A: SYSTEM PROMPT (lo que el LLM ve al arrancar)")
print("=" * 55)
print(agent.generate_system_prompt())

# --- PASO B: El LLM decide actuar (Thought + Action) ---
# En la realidad esta decision la toma el LLM a partir del mensaje del usuario.
# Aqui la hardcodeamos para hacer el flujo visible.
print("=" * 55)
print("PASO B: ACCION GENERADA POR EL LLM")
print("=" * 55)
print("  Thought: El usuario quiere el precio de Apple.")
print("  Action:  get_stock_price")
print("  Input:   AAPL")
print()

# --- PASO C: El motor ejecuta la herramienta y obtiene la Observation ---
observation = agent.handle_llm_call("get_stock_price", "AAPL")

print()
print("=" * 55)
print("PASO C: OBSERVATION (resultado que el LLM leeara)")
print("=" * 55)
print(observation)

# --- PASO D: Simular un error para ver como el agente lo maneja ---
print()
print("=" * 55)
print("PASO D: MANEJO DE ERROR (simbolo inexistente)")
print("=" * 55)
observation_error = agent.handle_llm_call("get_stock_price", "AAPLES")
print(observation_error)
print()
print("-> El LLM leera esta Observation y podra corregir su siguiente accion.")

PASO A: SYSTEM PROMPT (lo que el LLM ve al arrancar)
Eres un asistente con acceso a las siguientes herramientas:

- get_stock_price: Obtiene el precio actual de una accion. Si el simbolo no existe, sugiere opciones validas.
  Argumentos: {'symbol': {'type': 'string', 'description': 'Simbolo bursatil de la accion (ej: AAPL, TSLA, GOOGL)'}}
  Retorna: string

PASO B: ACCION GENERADA POR EL LLM
  Thought: El usuario quiere el precio de Apple.
  Action:  get_stock_price
  Input:   AAPL

[Agente] Ejecutando 'get_stock_price' con argumento: 'AAPL'

PASO C: OBSERVATION (resultado que el LLM leeara)
Observation: El precio de AAPL es 175.5 USD.

PASO D: MANEJO DE ERROR (simbolo inexistente)
[Agente] Ejecutando 'get_stock_price' con argumento: 'AAPLES'
Observation: Error: 'AAPLES' no encontrado. Intenta con: AAPL, TSLA, GOOGL

-> El LLM leera esta Observation y podra corregir su siguiente accion.
